In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, classification_report, roc_auc_score,
                            confusion_matrix, cohen_kappa_score)
from imblearn.over_sampling import SMOTE
import warnings
import time
import copy

warnings.filterwarnings('ignore')

df = pd.read_csv('Breast Cancer METABRIC.csv')

target_col = 'Tumor Stage'
original_count = len(df)
df = df[df[target_col] != 4].copy()
removed_count = original_count - len(df)
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)

missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

label_encoders = {}
if categorical_cols:
    for col in categorical_cols:
        le = LabelEncoder()
        unique_vals = X[col].unique()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

print(f"Feature matrix: {X.shape}")

# Feature selection
K_FEATURES = 15

selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(min(20, len(feature_scores))).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

selected_categorical = [col for col in selected_features if col in categorical_cols]
selected_numerical = [col for col in selected_features if col in numerical_cols]

# Data splitting
class_counts = y.value_counts()
print("\nNumber of samples per stage:")
for stage, count in class_counts.sort_index().items():
    print(f"  Stage {stage}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in y_train.value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")

# SMOTE - check minimum class sample count, adjust k_neighbors accordingly
min_class_samples = y_train.value_counts().min()
k_neighbors = min(5, min_class_samples - 1)  # k_neighbors must be less than min class count

if k_neighbors >= 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
else:
    print("Sample size too small for SMOTE, using original training set")
    X_train_bal, y_train_bal = X_train.values, y_train.values

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")

# Target variable encoding (for neural networks)
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train_bal)
y_test_encoded = target_encoder.transform(y_test)
num_classes = len(target_encoder.classes_)

print(f"\nNumber of classes: {num_classes}")
print(f"Class mapping: {dict(zip(target_encoder.classes_, range(num_classes)))}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)


# M2MASC - Modified Mixed Attention

class ModifiedMixedAttention(nn.Module):

    def __init__(self, channels, reduction_ratio=8):
        super(ModifiedMixedAttention, self).__init__()

        # Channel attention
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, channels // reduction_ratio, 1),
            nn.ReLU(inplace=True),
            nn.Conv1d(channels // reduction_ratio, channels, 1),
            nn.Sigmoid()
        )

        # Spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv1d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )

    def forward(self, x):

        ca_weight = self.channel_attention(x)  # (batch_size, channels, 1)
        x = x * ca_weight

        max_pool = torch.max(x, dim=1, keepdim=True)[0]  # (batch_size, 1, seq_len)
        avg_pool = torch.mean(x, dim=1, keepdim=True)    # (batch_size, 1, seq_len)
        sa_input = torch.cat([max_pool, avg_pool], dim=1)  # (batch_size, 2, seq_len)
        sa_weight = self.spatial_attention(sa_input)  # (batch_size, 1, seq_len)
        x = x * sa_weight

        return x


# Search Optimizer combining Search and Rescue Optimization and Spider Monkey Optimization

class SearchOptimizer:

    def __init__(self,
                 population_size=20,
                 max_iterations=30,
                 early_stop_patience=5):
        self.population_size = population_size
        self.max_iterations = max_iterations
        self.early_stop_patience = early_stop_patience

        self.param_space = {
            'cnn_filters': [
                [8, 16, 32],
                [16, 32, 64],
                [8, 16, 32, 64],
                [16, 32, 64, 128],
                [8, 16, 32, 64, 128]
            ],
            'lstm_hidden_size': [32, 64, 128],
            'dropout_rate': [0.1, 0.2, 0.3, 0.4, 0.5],
            'learning_rate': [0.0001, 0.0005, 0.001, 0.005],
        }

    def init_population(self):

        population = []
        for _ in range(self.population_size):
            individual = {
                'cnn_filters': np.random.choice(len(self.param_space['cnn_filters'])),
                'lstm_hidden_size': np.random.choice(self.param_space['lstm_hidden_size']),
                'dropout_rate': np.random.choice(self.param_space['dropout_rate']),
                'learning_rate': np.random.choice(self.param_space['learning_rate']),
                'fitness': 0.0
            }
            population.append(individual)
        return population

    def decode_individual(self, individual):

        params = {
            'cnn_filters': self.param_space['cnn_filters'][individual['cnn_filters']],
            'lstm_hidden_size': individual['lstm_hidden_size'],
            'dropout_rate': individual['dropout_rate'],
            'learning_rate': individual['learning_rate']
        }
        return params

    def evaluate_fitness(self, individual, num_features, num_classes,
                        train_loader, val_loader, device, max_epochs=8):

        params = self.decode_individual(individual)

        model = M2MASCCNNBiLSTM(
            num_features=num_features,
            num_classes=num_classes,
            cnn_filters=params['cnn_filters'],
            lstm_hidden_size=params['lstm_hidden_size'],
            dropout_rate=params['dropout_rate']
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

        best_acc = 0.0
        patience_counter = 0

        for epoch in range(max_epochs):

            model.train()
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    outputs = model(X_batch)
                    _, predicted = torch.max(outputs.data, 1)
                    total += y_batch.size(0)
                    correct += (predicted == y_batch).sum().item()

            acc = correct / total
            if acc > best_acc:
                best_acc = acc
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= 3:
                break

        return best_acc

    def update_position(self, population, global_best, iteration):
        # Update population positions based on paper formula 5
        # Combining internal and external guidance
        new_population = []

        for individual in population:

            PI0 = np.random.random()

            if PI0 < 0.5:
                # Internal guidance phase
                r1, r2, r3 = np.random.random(3)

                new_individual = {}
                for key in ['cnn_filters', 'lstm_hidden_size', 'dropout_rate', 'learning_rate']:
                    if key == 'cnn_filters':
                        # Discrete parameter: random selection
                        options = [individual[key], global_best[key],
                                 np.random.randint(0, len(self.param_space[key]))]
                        new_individual[key] = np.random.choice(options)
                    else:
                        # Continuous parameter: update by formula
                        current = individual[key]
                        best = global_best[key]

                        if isinstance(current, (int, np.integer)):
                            space = self.param_space[key]
                            idx = space.index(current) if current in space else 0
                            best_idx = space.index(best) if best in space else 0

                            new_idx = int(0.5 * (idx + r1 * (best_idx - idx)))
                            new_idx = max(0, min(len(space) - 1, new_idx))
                            new_individual[key] = space[new_idx]
                        else:
                            # Floating-point parameter
                            new_val = 0.5 * (current + r1 * (best - current))

                            new_individual[key] = np.clip(new_val,
                                                         min(self.param_space[key]),
                                                         max(self.param_space[key]))

                new_individual['fitness'] = 0.0
                new_population.append(new_individual)
            else:
                # External guidance phase: keep current position or apply small random mutation
                new_individual = individual.copy()

                if np.random.random() < 0.1:
                    mutate_key = np.random.choice(['dropout_rate', 'learning_rate'])
                    new_individual[mutate_key] = np.random.choice(self.param_space[mutate_key])

                new_population.append(new_individual)

        return new_population

    def optimize(self, num_features, num_classes, train_loader, val_loader, device):

        print(f"Population size: {self.population_size}")
        print(f"Maximum iterations: {self.max_iterations}")
        print(f"Hyperparameter space:")
        for param, values in self.param_space.items():
            if param == 'cnn_filters':
                print(f"  {param}: {len(values)} configurations")
            else:
                print(f"  {param}: {values}")

        population = self.init_population()
        global_best = None
        global_best_fitness = 0.0
        no_improve_count = 0

        for iteration in range(self.max_iterations):
            print(f"\nIteration {iteration + 1}/{self.max_iterations}")

            for idx, individual in enumerate(population):
                if individual['fitness'] == 0.0:
                    fitness = self.evaluate_fitness(
                        individual, num_features, num_classes,
                        train_loader, val_loader, device, max_epochs=15
                    )
                    individual['fitness'] = fitness

                    if fitness > global_best_fitness:
                        global_best_fitness = fitness
                        global_best = individual.copy()
                        no_improve_count = 0
                        print(f"  Individual {idx+1}: accuracy={fitness:.4f} [new best!]")
                    else:
                        print(f"  Individual {idx+1}: accuracy={fitness:.4f}")

            print(f"  Current best accuracy: {global_best_fitness:.4f}")

            no_improve_count += 1
            if no_improve_count >= self.early_stop_patience:
                print(f"\nEarly stopping: no improvement for {self.early_stop_patience} rounds")
                break

            if iteration < self.max_iterations - 1:
                population = self.update_position(population, global_best, iteration)

        best_params = self.decode_individual(global_best)

        print(f"\nSearch optimization complete")
        print(f"Best validation accuracy: {global_best_fitness:.4f}")
        print(f"Best hyperparameters:")
        for param, value in best_params.items():
            print(f"  {param}: {value}")

        return best_params


# M2MASC-enabled CNN-BiLSTM

class M2MASCCNNBiLSTM(nn.Module):

    def __init__(self,
                 num_features,
                 num_classes,
                 cnn_filters=[16, 32, 64],
                 lstm_hidden_size=64,
                 dropout_rate=0.2):
        super(M2MASCCNNBiLSTM, self).__init__()

        self.num_features = num_features
        self.num_classes = num_classes

        # 1. CNN layers
        self.conv_layers = nn.ModuleList()
        in_channels = 1
        for out_channels in cnn_filters:
            self.conv_layers.append(
                nn.Sequential(
                    nn.Conv1d(
                        in_channels=in_channels,
                        out_channels=out_channels,
                        kernel_size=3,
                        padding=1
                    ),
                    nn.ReLU(),
                    nn.MaxPool1d(kernel_size=2, stride=1, padding=1),
                    nn.Dropout(dropout_rate)
                )
            )
            in_channels = out_channels

        # 2. Modified Mixed Attention
        self.attention = ModifiedMixedAttention(
            channels=cnn_filters[-1],
            reduction_ratio=8
        )

        # 3. BiLSTM
        self.bilstm = nn.LSTM(
            input_size=cnn_filters[-1],
            hidden_size=lstm_hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0 if dropout_rate == 0 else dropout_rate
        )

        # 4. Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(lstm_hidden_size * 2, lstm_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(lstm_hidden_size, num_classes)
        )

    def forward(self, x):

        x = x.unsqueeze(1)  # (batch_size, 1, num_features)

        for conv_layer in self.conv_layers:
            x = conv_layer(x)

        x = self.attention(x)  # (batch_size, channels, seq_len)

        x = x.transpose(1, 2)  # (batch_size, seq_len, channels)

        lstm_out, (h_n, c_n) = self.bilstm(x)

        forward_hidden = h_n[-2, :, :]
        backward_hidden = h_n[-1, :, :]
        last_output = torch.cat([forward_hidden, backward_hidden], dim=1)

        logits = self.fc_layers(last_output)

        return logits


# Prepare PyTorch datasets

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TabularDataset(X_train_scaled, y_train_encoded)
test_dataset = TabularDataset(X_test_scaled, y_test_encoded)

# Validation split for search optimizer
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_train_scaled, y_train_encoded,
    test_size=0.2,
    random_state=42
)

train_opt_dataset = TabularDataset(X_train_opt, y_train_opt)
val_opt_dataset = TabularDataset(X_val_opt, y_val_opt)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

train_opt_loader = DataLoader(train_opt_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_opt_loader = DataLoader(val_opt_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Search Optimizer for hyperparameter tuning
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\ndevice: {device}")

USE_SEARCH_OPTIMIZER = True

if USE_SEARCH_OPTIMIZER:
    search_optimizer = SearchOptimizer(
        population_size=5,
        max_iterations=8,
        early_stop_patience=3
    )

    best_hyperparams = search_optimizer.optimize(
        num_features=len(selected_features),
        num_classes=num_classes,
        train_loader=train_opt_loader,
        val_loader=val_opt_loader,
        device=device
    )
else:
    best_hyperparams = {
        'cnn_filters': [8, 16, 32, 64, 128],
        'lstm_hidden_size': 64,
        'dropout_rate': 0.3,
        'learning_rate': 0.001
    }
    print("\nUsing manually configured hyperparameters:")
    for param, value in best_hyperparams.items():
        print(f"  {param}: {value}")

# Train final model with best hyperparameters
final_model = M2MASCCNNBiLSTM(
    num_features=len(selected_features),
    num_classes=num_classes,
    cnn_filters=best_hyperparams['cnn_filters'],
    lstm_hidden_size=best_hyperparams['lstm_hidden_size'],
    dropout_rate=best_hyperparams['dropout_rate']
).to(device)

total_params = sum(p.numel() for p in final_model.parameters())
trainable_params = sum(p.numel() for p in final_model.parameters() if p.requires_grad)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    final_model.parameters(),
    lr=best_hyperparams['learning_rate'],
    weight_decay=0.0001
)

NUM_EPOCHS = 200
PATIENCE = 20

best_test_acc = 0.0
patience_counter = 0
train_start_time = time.time()

for epoch in range(NUM_EPOCHS):

    final_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = final_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += y_batch.size(0)
        train_correct += (predicted == y_batch).sum().item()

    train_acc = train_correct / train_total
    avg_train_loss = train_loss / len(train_loader)

    final_model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = final_model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            test_total += y_batch.size(0)
            test_correct += (predicted == y_batch).sum().item()

    test_acc = test_correct / test_total

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Test Acc: {test_acc:.4f}")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0
        torch.save(final_model.state_dict(), 'best_m2masc_cnn_bilstm_breast_model.pth')
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

train_end_time = time.time()
training_time = train_end_time - train_start_time

print(f"Best test accuracy: {best_test_acc:.4f}")
print(f"Training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

final_model.load_state_dict(torch.load('best_m2masc_cnn_bilstm_breast_model.pth'))

# Cross-validation note: deep learning models require retraining for each fold
print(f"Current best test accuracy: {best_test_acc:.4f}")

# Feature importance analysis
print(feature_scores[feature_scores['feature'].isin(selected_features)].to_string(index=False))

# Model evaluation

def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    # AUC
    try:
        if num_classes == 2:
            auc = roc_auc_score(all_labels, all_probs[:, 1])
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    # Specificity
    cm = confusion_matrix(all_labels, all_preds)
    if num_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0
        for i in range(num_classes):
            tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
            fp = np.sum(cm[:, i]) - cm[i, i]
            specificity += tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificity /= num_classes

    # Cohen's Kappa
    kappa = cohen_kappa_score(all_labels, all_preds)

    return accuracy, precision, recall, f1, specificity, kappa, auc, all_preds, all_labels


train_accuracy, train_precision, train_recall, train_f1, train_spec, train_kappa, train_auc, _, _ = evaluate_model(
    final_model, train_loader, device
)

test_accuracy, test_precision, test_recall, test_f1, test_spec, test_kappa, test_auc, y_test_pred, y_test_labels = evaluate_model(
    final_model, test_loader, device
)

print("=" * 80)
print(" " * 20 + "M2MASC-CNN-BiLSTM Model Performance Comparison")
print("=" * 80)
print(f"{'Metric':<20} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 80)
print(f"{'Accuracy':<20} {train_accuracy*100:>14.2f}% {test_accuracy*100:>14.2f}% {abs(train_accuracy-test_accuracy)*100:>14.2f}%")
print(f"{'Precision':<20} {train_precision*100:>14.2f}% {test_precision*100:>14.2f}% {abs(train_precision-test_precision)*100:>14.2f}%")
print(f"{'Recall':<20} {train_recall*100:>14.2f}% {test_recall*100:>14.2f}% {abs(train_recall-test_recall)*100:>14.2f}%")
print(f"{'F1 Score':<20} {train_f1*100:>14.2f}% {test_f1*100:>14.2f}% {abs(train_f1-test_f1)*100:>14.2f}%")
print(f"{'Specificity':<20} {train_spec*100:>14.2f}% {test_spec*100:>14.2f}% {abs(train_spec-test_spec)*100:>14.2f}%")
print(f"{'Cohen Kappa':<20} {train_kappa*100:>14.2f}% {test_kappa*100:>14.2f}% {abs(train_kappa-test_kappa)*100:>14.2f}%")
print(f"{'AUC':<20} {train_auc*100:>14.2f}% {test_auc*100:>14.2f}% {abs(train_auc-test_auc)*100:>14.2f}%")
print("=" * 80)

y_test_original = target_encoder.inverse_transform(y_test_labels)
y_test_pred_original = target_encoder.inverse_transform(y_test_pred)
print(classification_report(y_test_original, y_test_pred_original, zero_division=0, digits=4))